In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from xgboost import XGBRegressor
import warnings

warnings.filterwarnings("ignore")

# ۱. بارگذاری داده‌ها
file_path = r'second_stage_inputs\G12\dsas_g12_bearings_vibration_temp_output.xlsx'
df_raw = pd.read_excel(file_path)
df_raw['date'] = pd.to_datetime(df_raw['date'])
df_raw = df_raw.sort_values(by='date')

all_sensors = ['AssetID_12208', 'AssetID_9343', 'AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 'AssetID_9368', 'AssetID_9369', 'AssetID_9370']

# ۲. جدا سازی دیتای آموزشی (فرض می‌کنیم داده‌های قدیمی‌تر، وضعیت پایه سیستم هستند)
# اما کل این داده‌ها را برای آموزش استفاده نمی‌کنیم، ابتدا آن‌ها را پالایش می‌کنیم.
last_date = df_raw['date'].max()
split_date = last_date - pd.Timedelta(days=20)
raw_train_set = df_raw[df_raw['date'] <= split_date].copy()
test_set = df_raw[df_raw['date'] > split_date].copy()

# ۳. پالایش دیتای آموزش (ساختن Normal Profile)
# استفاده از DBSCAN برای شناسایی و حذف نقاطی که با رفتار کلی سیستم همخوانی ندارند
scaler = StandardScaler()
scaled_train = scaler.fit_transform(raw_train_set[all_sensors])

dbscan = DBSCAN(eps=0.8, min_samples=10) # پارامترها بسته به چگالی داده شما قابل تنظیم است

labels = dbscan.fit_predict(scaled_train)

# فقط داده‌هایی که در کلاستر اصلی هستند (Label != -1) به عنوان "دیتای سالم" انتخاب می‌شوند
normal_train_set = raw_train_set[labels != -1].copy()

print(f"تعداد کل داده‌های آموزشی: {len(raw_train_set)}")
print(f"تعداد داده‌های سالم تایید شده برای آموزش: {len(normal_train_set)}")
print(f"میزان داده‌های پرت حذف شده: {len(raw_train_set) - len(normal_train_set)} ردیف")

# ۴. آموزش مدل‌ها بر اساس "فقط داده‌های سالم"
models = {}
for target in all_sensors:
    features = [s for s in all_sensors if s != target]
    model = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42)
    model.fit(normal_train_set[features], normal_train_set[target])
    models[target] = model

# ۵. پایش انحراف در ۲۰ روز اخیر
all_errors = pd.DataFrame(index=test_set.index)
for target in all_sensors:
    features = [s for s in all_sensors if s != target]
    preds = models[target].predict(test_set[features])
    # محاسبه انحراف (تفاضل مقدار واقعی از مقداری که مدلِ "سالم" پیش‌بینی می‌کند)
    all_errors[f'err_{target}'] = np.abs(test_set[target] - preds)

# ۶. محاسبه شاخص انحراف کل سیستم (System Deviation Index)
test_set['System_Deviation_Index'] = all_errors.mean(axis=1)

# محاسبه آستانه هشدار بر اساس انحرافات دیتای سالم (Training Error Threshold)
# اگر انحراف در تست، خیلی بیشتر از انحراف در زمان آموزش باشد، یعنی سیستم غیرعادی است.
train_preds_all = []
for target in all_sensors:
    features = [s for s in all_sensors if s != target]
    train_preds = models[target].predict(normal_train_set[features])
    train_preds_all.append(np.abs(normal_train_set[target] - train_preds))

mean_train_error = np.mean(train_preds_all)
std_train_error = np.std(train_preds_all)
threshold = mean_train_error + (3*std_train_error) # آستانه ۳ سیگما

test_set['Is_Anomaly'] = test_set['System_Deviation_Index'] > threshold

# ۷. ذخیره خروجی
output_path = r'outputs\G12\dsas_g12_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g12_bearings_vibration_temp_deviation_monitoring_output3.xlsx'
test_set.to_excel(output_path, index=False)
print(f"پایش با مدل پالایش شده انجام و در {output_path} ذخیره شد.")

تعداد کل داده‌های آموزشی: 5567
تعداد داده‌های سالم تایید شده برای آموزش: 5368
میزان داده‌های پرت حذف شده: 199 ردیف
پایش با مدل پالایش شده انجام و در outputs\G12\dsas_g12_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g12_bearings_vibration_temp_deviation_monitoring_output3.xlsx ذخیره شد.
